# 04 — Test & Evaluate
## DREBIN — Android Malware Permissions

Evaluate on test set + IEEE figures.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
from pathlib import Path
from src.io.registry import get_dataset, ACTIVE_DATASETS
from src.io.paths import ensure_dataset_dirs, data_processed_dir, models_dir, reports_figures_dir

DATASET_ID = "drebin"
cfg = get_dataset(DATASET_ID)
paths = ensure_dataset_dirs(cfg)
np.random.seed(42)
print(cfg.display_name)

from src.io.loaders import load_processed
from src.core.model import RandomForestModel
from src.core.evaluation import compute_metrics
from src.core.metrics import positive_class_proba
from src.core.ieee_style import ieee_bar_chart, ieee_confusion_matrix, ieee_roc_curve
from sklearn.metrics import confusion_matrix, roc_curve, auc

data = load_processed(DATASET_ID)
rf = RandomForestModel.load(str(models_dir(DATASET_ID)/'rf_shap_model.pkl'))
y_pred = rf.predict(data.X_test)
y_proba = positive_class_proba(rf.predict_proba(data.X_test))
metrics = compute_metrics(data.y_test, y_pred, y_proba, cfg.display_name)
print(metrics)


In [ ]:

stem = reports_figures_dir()/DATASET_ID
ieee_bar_chart(['Acc','Prec','Rec','F1','AUC'],
    [metrics['accuracy'],metrics['precision'],metrics['recall'],metrics['f1'],metrics['auc_roc']],
    f'{cfg.display_name} Test Metrics', 'Score', stem/'bar_metrics')
cm = confusion_matrix(data.y_test, y_pred, labels=[0,1])
ieee_confusion_matrix(cm, list(cfg.class_names), 'Confusion Matrix', stem/'confusion_matrix')
fpr,tpr,_ = roc_curve(data.y_test, y_proba)
ieee_roc_curve(fpr, tpr, auc(fpr,tpr), 'ROC', stem/'roc_curve')
print('IEEE figures saved to', stem)
